In [ ]:
import tensorflow as tf
import numpy as np

gpu = tf.config.experimental.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpu[0], True)
tf.keras.utils.set_random_seed(42)

# This is a TF-rUnet model code

## Resnet Block RSB

In [ ]:
class ResBlock1D(tf.keras.layers.Layer):
    def __init__(self, covfilter, resnum, covks=5, covstrides=1, rb_trainable=True, **kwargs):
        super(ResBlock1D, self).__init__(name=f'rb{resnum}_block', **kwargs)
        self.rescov = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=3,
            strides=covstrides,
            padding='causal',
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_rescov'
        )

        self.res1_bn1 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res1_bn1')
        self.res1_act1 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res1_act1')
        self.res1_cov1 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            padding='causal',
            dilation_rate=2,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res1_cov1'
        )
        self.res1_bn2 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res1_bn2')
        self.res1_act2 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res1_act2')        
        self.res1_cov2 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            padding='causal',
            dilation_rate=4,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res1_cov2'
        )
        self.add1 = tf.keras.layers.Add(name=f'rb{resnum}_res1_add')

        self.res2_bn1 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res2_bn1')
        self.res2_act1 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res2_act1')
        self.res2_cov1 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks+2,
            dilation_rate=6,
            padding='causal',
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res2_cov1'
        )
        self.res2_bn2 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res2_bn2')
        self.res2_act2 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res2_act2')        
        self.res2_cov2 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks+2,
            dilation_rate=8,
            padding='causal',
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res2_cov2'
        )
        self.add2 = tf.keras.layers.Add(name=f'rb{resnum}_res2_add')

        # self.resbn = tf.keras.layers.BatchNormalization(name=f'rb{resnum}_resbn')
        # self.resact2 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_resact2') 


    def call(self, x):
        rescov = self.rescov(x)
        
        res1_bn1 = self.res1_bn1(rescov)
        res1_act1 = self.res1_act1(res1_bn1)
        res1_cov1 = self.res1_cov1(res1_act1)
        res1_bn2 = self.res1_bn2(res1_cov1)
        res1_act2 = self.res1_act2(res1_bn2)
        res1_cov2 = self.res1_cov2(res1_act2)        
        res1_add = self.add1([rescov, res1_cov2])

        res2_bn1 = self.res2_bn1(res1_add)
        res2_act1 = self.res2_act1(res2_bn1)
        res2_cov1 = self.res2_cov1(res2_act1)
        res2_bn2 = self.res2_bn2(res2_cov1)
        res2_act2 = self.res2_act2(res2_bn2)
        res2_cov2 = self.res2_cov2(res2_act2)        
        res2_add = self.add2([res1_add, res2_cov2])

        # resbn = self.resbn(res2_add)
        # resact2 = self.resact2(resbn)
        return res2_add

## FFTLayer

In [ ]:
class FFTLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FFTLayer, self).__init__(**kwargs)

    def call(self, inputs):
        fft_result = tf.signal.fft(tf.cast(inputs[:,:,0], tf.complex64))
        amp_norm = np.ones((1,1250))/625
        amp_norm[:,0] = 0
        real_part = tf.math.real(fft_result)*amp_norm
        imag_part = tf.math.imag(fft_result)*amp_norm
        return tf.concat([real_part[:,:,tf.newaxis], imag_part[:,:,tf.newaxis]], axis=-1)

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, 2 * channel)
        return (input_shape[0], input_shape[1], 2 * input_shape[2])

## IFFTLayer

In [ ]:
class IFFTLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(IFFTLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # 将输入分为实部和虚部
        real_part = inputs[..., 0]
        imag_part = inputs[..., 1]
        # 组合成复数
        complex_input = tf.complex(real_part, imag_part)
        # 对每个 channel 进行 IFFT 变换
        ifft_result = tf.signal.ifft(complex_input)
        return tf.math.real(ifft_result)[:,:,tf.newaxis]

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, channel)
        return (input_shape[0], input_shape[1], input_shape[2] // 2)

## Cross Time-Frequency Attention TFCA

In [ ]:
class CrossTFAttention(tf.keras.layers.Layer):
    def __init__(self, attnum, num_head, key_dim, output_shape, **kwargs):
        super(CrossTFAttention, self).__init__(name=f'crftatt{attnum}_block', **kwargs)
        self.frqcov = tf.keras.layers.Conv1D(filters=2,
            kernel_size=5,
            padding='same',
            name=f'crftatt{attnum}_frqcov')

        self.frqtimcov = tf.keras.layers.Conv1D(filters=output_shape,
            kernel_size=1,
            padding='same',
            name=f'crftatt{attnum}_frqtimcov')
        
        self.ifft = IFFTLayer()
        self.mhatt = tf.keras.layers.MultiHeadAttention(num_heads=num_head,key_dim=key_dim,output_shape=output_shape,name=f'crftatt{attnum}_mhatt')
        self.attadd = tf.keras.layers.Add(name=f'crftatt{attnum}_add')
        
    def call(self, frq, sig):
        frq_cplx = self.frqcov(frq)
        frq_sig = self.ifft(frq_cplx)
        ftime_sig = self.frqtimcov(frq_sig)
        
        timfreq_att = self.mhatt(query=ftime_sig,value=sig,key=sig)
        resadd_att = self.attadd([sig,timfreq_att,frq_sig])

        return resadd_att 

## TF-rUnet Foundation Model

In [ ]:
def FMAutoEncoder(fs: int=125) -> tf.keras.Model:
    # =========================================== signal layers resUnet ===============================================
    ipl = tf.keras.Input((10*fs,1))
    fftipl = FFTLayer()(ipl)
    # =========================================== signal Encoder ======================================================
    res1 = ResBlock1D(16,1)(ipl)
    res1_mp = tf.keras.layers.MaxPool1D(pool_size=5,name="res1_mp")(res1)
    res2 = ResBlock1D(32,2,covks=5)(res1_mp)
    res2_mp = tf.keras.layers.MaxPool1D(pool_size=5,name="res2_mp")(res2)
    res3 = ResBlock1D(64,3,covks=5)(res2_mp)
    res3_mp = tf.keras.layers.MaxPool1D(pool_size=5,name="res3_mp")(res3)
    res4 = ResBlock1D(128,4,covks=5)(res3_mp)
    res4_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res4_mp")(res4)
    res_enc = ResBlock1D(256,9,covks=5)(res4_mp)
       
    # ============================================ FFT Encoder =========================================================
    res11 = ResBlock1D(16,11)(fftipl)
    res11_mp = tf.keras.layers.MaxPool1D(pool_size=5,name="res11_mp")(res11)
    res12 = ResBlock1D(32,12,covks=5)(res11_mp)
    res12_mp = tf.keras.layers.MaxPool1D(pool_size=5,name="res12_mp")(res12)
    res13 = ResBlock1D(64,13,covks=5)(res12_mp)
    res13_mp = tf.keras.layers.MaxPool1D(pool_size=5,name="res13_mp")(res13)
    res14 = ResBlock1D(128,14,covks=5)(res13_mp)
    res14_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res14_mp")(res14)
    res_enc_fft = ResBlock1D(256,19,covks=5)(res14_mp) 
        
    # ============================================= FrqTime Cross Attention ============================================
    ctfa_scale1 = CrossTFAttention(attnum=1, num_head=2, key_dim=8, output_shape=16)(frq=res11,sig=res1)
    ctfa_scale2 = CrossTFAttention(attnum=2, num_head=2, key_dim=8, output_shape=32)(frq=res12,sig=res2)
    ctfa_scale3 = CrossTFAttention(attnum=3, num_head=2, key_dim=8, output_shape=64)(frq=res13,sig=res3)
    ctfa_scale4 = CrossTFAttention(attnum=4, num_head=2, key_dim=8, output_shape=128)(frq=res14,sig=res4)
    ctfa_scale_enc = CrossTFAttention(attnum=5, num_head=2, key_dim=8, output_shape=256)(frq=res_enc_fft,sig=res_enc)

    # =========================================== signal Decoder ======================================================
    res4_up = tf.keras.layers.UpSampling1D(size=2,name="res4_up")(ctfa_scale_enc)
    res4_cc = tf.keras.layers.concatenate([res4_up,ctfa_scale4],name="res4_cc")
    res4_rb = ResBlock1D(128,5,covks=5)(res4_cc)
    res3_up = tf.keras.layers.UpSampling1D(size=5,name="res3_up")(res4_rb)
    res3_cc = tf.keras.layers.concatenate([res3_up,ctfa_scale3],name="res3_cc")
    res3_rb = ResBlock1D(64,6,covks=5)(res3_cc)
    res2_up = tf.keras.layers.UpSampling1D(size=5,name="res2_up")(res3_rb)
    res2_cc = tf.keras.layers.concatenate([res2_up,ctfa_scale2],name="res2_cc")
    res2_rb = ResBlock1D(32,7,covks=5)(res2_cc)
    res1_up = tf.keras.layers.UpSampling1D(size=5,name="res1_up")(res2_rb)
    res1_cc = tf.keras.layers.concatenate([res1_up,ctfa_scale1],name="res1_cc")
    res1_rb = ResBlock1D(16,8,covks=5)(res1_cc)

    runet_sig_output = tf.keras.layers.Conv1D(filters=1,kernel_size=1,padding='same',kernel_initializer=tf.keras.initializers.HeNormal(),name="runet_sig_output")(res1_rb)
  
    # ============================================ FFT Decoder =========================================================
    res14_up = tf.keras.layers.UpSampling1D(size=2,name="res14_up")(res_enc_fft)
    res14_cc = tf.keras.layers.concatenate([res14_up,res14],name="res14_cc")
    res14_rb = ResBlock1D(128,15,covks=5)(res14_cc)
    res13_up = tf.keras.layers.UpSampling1D(size=5,name="res13_up")(res14_rb)
    res13_cc = tf.keras.layers.concatenate([res13_up,res13],name="res13_cc")
    res13_rb = ResBlock1D(64,16,covks=5)(res13_cc)
    res12_up = tf.keras.layers.UpSampling1D(size=5,name="res12_up")(res13_rb)
    res12_cc = tf.keras.layers.concatenate([res12_up,res12],name="res12_cc")
    res12_rb = ResBlock1D(32,17,covks=5)(res12_cc)
    res11_up = tf.keras.layers.UpSampling1D(size=5,name="res11_up")(res12_rb)
    res11_cc = tf.keras.layers.concatenate([res11_up,res11],name="res11_cc")
    res11_rb = ResBlock1D(16,18,covks=5)(res11_cc)

    runet_fft_output = tf.keras.layers.Conv1D(filters=2,kernel_size=1,padding='same',kernel_initializer=tf.keras.initializers.HeNormal(),name="runet_fft_output")(res11_rb)
    runet_ifft_output = IFFTLayer()(runet_fft_output)

    sig_merg_concat = tf.keras.layers.concatenate([runet_sig_output,runet_ifft_output],name="sig_merg_concat")
    runet_output = tf.keras.layers.Conv1D(filters=1,kernel_size=3,padding='same',kernel_initializer=tf.keras.initializers.HeNormal(),name="runet_output")(sig_merg_concat)

    runet = tf.keras.Model(inputs=ipl,outputs=runet_output)

    return runet

# This is a TF-rUnet structure used in MP-DANN

## GRL layer

In [ ]:
class GradientReversalLayer(tf.keras.layers.Layer):
    """The gradient reversal layer is a layer that multiplies the gradient by a negative constant during
    backpropagation.

    Args:
      lambda_: Float32, the constant by which the gradient is multiplied. It should be a negative number.
    """
    def __init__(self, lambda_: float = -0.005):
        super().__init__(trainable=False, name="gradient_reversal_layer")
        self.lambda_ = tf.constant(lambda_, dtype=tf.float32)  # Normally, a negative value

    def call(self, x, **kwargs):
        return self.grad_reversed(x)

    @tf.custom_gradient
    def grad_reversed(self, x):
        """
        It returns input and a custom gradient function.

        Args:
          x: The input tensor.

        Returns:
          the input x and the custom gradient function.
        """
        def custom_gradient(dy):
            return self.lambda_ * dy

        return x, custom_gradient

    def get_config(self):
        config = super().get_config().copy()
        config.update({
            "lambda": float(self.lambda_.numpy())
        })
        return config

## Public view structure

In [ ]:
def EmoEncoder_simple(fm: tf.keras.Model, fs: int=125, fm_tr = False) -> tf.keras.Model :
    fm.trainable = fm_tr
    rsb1 = fm.get_layer(name="crftatt1_block").output
    rsb2 = fm.get_layer(name="crftatt2_block").output
    rsb3 = fm.get_layer(name="crftatt3_block").output
    rsb4 = fm.get_layer(name="crftatt4_block").output

    rsb11 = fm.get_layer(name="rb11_block").output
    rsb12 = fm.get_layer(name="rb12_block").output
    rsb13 = fm.get_layer(name="rb13_block").output
    rsb14 = fm.get_layer(name="rb14_block").output
    
    up4 = tf.keras.layers.UpSampling1D(size=125,name="up4")(rsb4)
    up3 = tf.keras.layers.UpSampling1D(size=25,name="up3")(rsb3)
    up2 = tf.keras.layers.UpSampling1D(size=5,name="up2")(rsb2)

    up14 = tf.keras.layers.UpSampling1D(size=125,name="up14")(rsb14)
    up13 = tf.keras.layers.UpSampling1D(size=25,name="up13")(rsb13)
    up12 = tf.keras.layers.UpSampling1D(size=5,name="up12")(rsb12)

    tenc_cc = tf.keras.layers.concatenate([rsb1,up2,up3,up4],name="tenc_cc")
    fenc_cc = tf.keras.layers.concatenate([rsb11,up12,up13,up14],name="fenc_cc")

    sig_deco = tf.keras.layers.MultiHeadAttention(num_heads=2,key_dim=8,kernel_initializer="he_normal",name="sig_deco")(tenc_cc,tenc_cc)
    sig_dec_res = tf.keras.layers.Add(name="sig_dec_res")([tenc_cc,sig_deco])
    sig_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='sig_deco_gap')(sig_dec_res)
    sig_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='sig_deco_gmp')(sig_dec_res)
    sig_deco_cc = tf.keras.layers.concatenate([sig_deco_gap,sig_deco_gmp],name="sig_deco_cc")
    sig_dec_f = tf.keras.layers.Dense(units=16,kernel_initializer="he_normal",name="sig_dec_f")(sig_deco_cc)
    sig_dec_gelu = tf.keras.layers.Activation(activation='gelu',name="sig_dec_gelu")(sig_dec_f)

    frq_deco = tf.keras.layers.MultiHeadAttention(num_heads=2,key_dim=8,kernel_initializer="he_normal",name="frq_deco")(fenc_cc,fenc_cc)
    frq_dec_res = tf.keras.layers.Add(name="frq_dec_res")([fenc_cc,frq_deco])
    frq_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='frq_deco_gap')(frq_dec_res)
    frq_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='frq_deco_gmp')(frq_dec_res)
    frq_deco_cc = tf.keras.layers.concatenate([frq_deco_gap,frq_deco_gmp],name="frq_deco_cc")
    frq_dec_f = tf.keras.layers.Dense(units=16,kernel_initializer="he_normal",name="frq_dec_f")(frq_deco_cc)
    frq_dec_gelu = tf.keras.layers.Activation(activation='gelu',name="frq_dec_gelu")(frq_dec_f)

    ft_cc = tf.keras.layers.concatenate([sig_dec_gelu,frq_dec_gelu],name="ft_cc")

    emo_f = tf.keras.layers.Dense(units=16,kernel_initializer="he_normal",name="emo_f")(ft_cc)
    emo_gelu = tf.keras.layers.Activation(activation='gelu',name="emo_gelu")(emo_f)
    emo_cls = tf.keras.layers.Dense(units=2,activation='softmax',name="emo_cls")(emo_gelu)

    sub_cls = tf.keras.layers.Dense(units=30,activation='softmax',name="sub_cls")(ft_cc)
    emomodel = tf.keras.Model(inputs=fm.input,outputs=[emo_cls,sub_cls])

    return emomodel

## Private view structure

In [ ]:
def EmoEncoder_grl(fm: tf.keras.Model, fs: int=125, fm_tr = False) -> tf.keras.Model :
    fm.trainable = fm_tr
    rsb1 = fm.get_layer(name="crftatt1_block").output
    rsb2 = fm.get_layer(name="crftatt2_block").output
    rsb3 = fm.get_layer(name="crftatt3_block").output
    rsb4 = fm.get_layer(name="crftatt4_block").output

    rsb11 = fm.get_layer(name="rb11_block").output
    rsb12 = fm.get_layer(name="rb12_block").output
    rsb13 = fm.get_layer(name="rb13_block").output
    rsb14 = fm.get_layer(name="rb14_block").output
    
    up4 = tf.keras.layers.UpSampling1D(size=125,name="up4")(rsb4)
    up3 = tf.keras.layers.UpSampling1D(size=25,name="up3")(rsb3)
    up2 = tf.keras.layers.UpSampling1D(size=5,name="up2")(rsb2)

    up14 = tf.keras.layers.UpSampling1D(size=125,name="up14")(rsb14)
    up13 = tf.keras.layers.UpSampling1D(size=25,name="up13")(rsb13)
    up12 = tf.keras.layers.UpSampling1D(size=5,name="up12")(rsb12)

    tenc_cc = tf.keras.layers.concatenate([rsb1,up2,up3,up4],name="tenc_cc")
    fenc_cc = tf.keras.layers.concatenate([rsb11,up12,up13,up14],name="fenc_cc")

    sig_deco = tf.keras.layers.MultiHeadAttention(num_heads=2,key_dim=8,kernel_initializer="he_normal",name="sig_deco")(tenc_cc,tenc_cc)
    sig_dec_res = tf.keras.layers.Add(name="sig_dec_res")([tenc_cc,sig_deco])
    sig_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='sig_deco_gap')(sig_dec_res)
    sig_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='sig_deco_gmp')(sig_dec_res)
    sig_deco_cc = tf.keras.layers.concatenate([sig_deco_gap,sig_deco_gmp],name="sig_deco_cc")
    sig_dec_f = tf.keras.layers.Dense(units=16,kernel_initializer="he_normal",name="sig_dec_f")(sig_deco_cc)
    sig_dec_gelu = tf.keras.layers.Activation(activation='gelu',name="sig_dec_gelu")(sig_dec_f)

    frq_deco = tf.keras.layers.MultiHeadAttention(num_heads=2,key_dim=8,kernel_initializer="he_normal",name="frq_deco")(fenc_cc,fenc_cc)
    frq_dec_res = tf.keras.layers.Add(name="frq_dec_res")([fenc_cc,frq_deco])
    frq_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='frq_deco_gap')(frq_dec_res)
    frq_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='frq_deco_gmp')(frq_dec_res)
    frq_deco_cc = tf.keras.layers.concatenate([frq_deco_gap,frq_deco_gmp],name="frq_deco_cc")
    frq_dec_f = tf.keras.layers.Dense(units=16,kernel_initializer="he_normal",name="frq_dec_f")(frq_deco_cc)
    frq_dec_gelu = tf.keras.layers.Activation(activation='gelu',name="frq_dec_gelu")(frq_dec_f)

    ft_cc = tf.keras.layers.concatenate([sig_dec_gelu,frq_dec_gelu],name="ft_cc")

    emo_f = tf.keras.layers.Dense(units=16,kernel_initializer="he_normal",name="emo_f")(ft_cc)
    emo_gelu = tf.keras.layers.Activation(activation='gelu',name="emo_gelu")(emo_f)
    emo_cls = tf.keras.layers.Dense(units=2,activation='softmax',name="emo_cls")(emo_gelu)

    grl = GradientReversalLayer()(ft_cc)
    sub_cls = tf.keras.layers.Dense(units=30,activation='softmax',name="sub_cls")(grl)
    emomodel = tf.keras.Model(inputs=fm.input,outputs=[emo_cls,sub_cls])

    return emomodel